# Imaging: reward-aligned responses and most responsive cells

Everything is on **pyControl time (s)**, using the same rsync alignment as the photometry, while keeping **frame numbers**: frame `i` of `F.npy` has pyControl time `time_s[i]` in `run-XX/frame_times.npz`.

1. **Frame times.** For each run, match the raw TIFF, the acquisition's DAQ file (`file_*.h5`: frame clock and rsync pulses) and the behaviour `.tsv` whose rsync pulses align with the DAQ's, and save `frame_times.npz` beside the run in derivatives. Written once per run; both destriping variants share it.
2. **Reward-aligned responses.** Per variant: each cell's mean response around the outcome (reward onset, or no-reward cue onset for 0 pulses) per number of reward pulses, as robust z-scores (default) or raw F and ΔF/F0 (`MEASURES`) of the green and red channels; then mean ± SEM across all cells of all runs.
3. **Most responsive cells.** Per variant and run: the `N_TOP` best cells by Suite2p's classifier probability p(cell) and, separately, by largest green ΔF/F0, their traces with the outcomes marked, and where they are.

Needs the extracted planes (batch + Cellpose extraction in the mini2p repo) and `uv sync --extra funcimg --extra notebooks`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from matplotlib.backends.backend_pdf import PdfPages
from mini2p_preprocessing.progress import ProgressLog

import src.behavior_visualization.plot_style  # noqa: F401
from src.behavior_analysis.outcome_events import OutcomeParseError
from src.behavior_import.session_order import session_number
from src.funcimg_analysis.collect import collect_pulse_responses, top_cells, variant_runs
from src.funcimg_analysis.frame_times import (
    FRAME_TIMES_FILE,
    RunMismatchError,
    load_frame_times,
    write_frame_times,
)
from src.funcimg_import.runs import session_run_files
from src.funcimg_visualization.plot_funcimg_responses import (
    plot_pulse_responses,
    plot_top_cells,
    pulse_responses_png_name,
    top_cells_pdf_name,
)
from src.settings import REPO_ROOT, get_settings

EXPERIMENT = "3x3_field_magnitude_bandit"
COHORT = "cohort-01"
SUBJECT = "sub-03_id-MY_71_R"
SESSION_FIRST, SESSION_LAST = 224, 224  # one session: give the same number twice
RUNS = ("run-01",)  # every run; or only these runs of each session, e.g. ("run-02",)
SOURCE_SUBSTRING = "71rtask"  # picks the imaging TIFFs, as in the batch preprocessing
VARIANTS = ("without_destriping", "with_destriping")
WINDOW_S = (-10.0, 30.0)  # around the outcome; rounded to whole frames
PHASES = ("task",)  # add "port_finding" to include port-finding trials
X_UNITS = "s"  # "s" or "frames", for the pooled response figures
TRACES_X_UNITS = "frames"  # "s" or "frames", for the top-cell traces
TRACES_MIDDLE_FRAMES = 8000  # top-cell traces show only the middle frames of each run; None = all
MEASURES = ("F z", "dF/F0 z")  # robust z, baseline-subtracted; or ("F", "dF/F0") for raw
# z needs frames before the outcome: with a WINDOW_S starting at 0, use the raw measures
MEASURE_TAG = "-".join(m.replace("/", "").replace(" ", "") for m in MEASURES)  # in file names
CLIP_PERCENTILES = (1, 99)  # top-cell traces clipped to this range per ROI; None = unclipped
N_TOP = 30
RANKINGS = ("p_cell", "max_dff")  # one PDF per ranking
OVERWRITE_FRAME_TIMES = False

data_root = get_settings().data_root
RAW = data_root / EXPERIMENT / "rawdata" / COHORT / SUBJECT
DERIVATIVES = data_root / EXPERIMENT / "derivatives" / COHORT / SUBJECT
FIGURES = REPO_ROOT / "results" / "figures" / "funcimg" / COHORT / SUBJECT
FIGURES.mkdir(parents=True, exist_ok=True)
SCOPE = f"sessions {SESSION_FIRST}-{SESSION_LAST}" + (f", {', '.join(RUNS)}" if RUNS else "")
FILE_TAG = f"ses-{SESSION_FIRST}-{SESSION_LAST}" + (f"_{'_'.join(RUNS)}" if RUNS else "")
# Folders without a session number (e.g. ses-baseplating) are skipped.
sessions = sorted(
    (
        p
        for p in RAW.glob("ses-*")
        if session_number(p.name) is not None
        and SESSION_FIRST <= session_number(p.name) <= SESSION_LAST
        and (p / "funcimg").is_dir()
    ),
    key=lambda p: session_number(p.name),
)
print(f"{len(sessions)} imaging sessions in {RAW}")

## 1. Frame times

Writes `derivatives/.../funcimg/run-XX/frame_times.npz` for every run the batch has preprocessed. Before writing, the run's TIFF is checked against the batch's own `source_tiff.json`, so a run folder can never be paired with another acquisition's frame clock or behaviour. A frame's time is the start of its acquisition (frame-clock rising edge). Existing files are kept unless `OVERWRITE_FRAME_TIMES`. The table shows how well each run's rsync pulses matched; `fraction_matched` should be close to 1 and the interval discrepancy a few ms at most.

In [ ]:
rows, problems = [], []
progress = ProgressLog("Frame times", len(sessions), print)
for session_dir in sessions:
    try:
        runs = session_run_files(session_dir, SOURCE_SUBSTRING)
        if RUNS is not None:
            runs = [run for run in runs if run.run_name in RUNS]
    except ValueError as error:
        problems.append((session_dir.name, "-", str(error)))
        progress.step_done(succeeded=False)
        continue
    for run in runs:
        run_dir = DERIVATIVES / session_dir.name / "funcimg" / run.run_name
        path = run_dir / FRAME_TIMES_FILE
        try:
            if OVERWRITE_FRAME_TIMES or not path.exists():
                write_frame_times(run, run_dir, overwrite=OVERWRITE_FRAME_TIMES)
            saved = load_frame_times(run_dir)
        except (RunMismatchError, FileNotFoundError) as error:
            problems.append((session_dir.name, run.run_name, str(error)))
            continue
        rows.append({
            "session": session_dir.name, "run": run.run_name, "tiff": run.tiff.name,
            "daq": run.h5.name, "behaviour": run.tsv.name, "frames": len(saved.time_s),
            "frames_with_time": int(pd.notna(saved.time_s).sum()),
            "fraction_matched": round(saved.quality["fraction_matched"], 3),
            "max_interval_discrepancy_ms": saved.quality["max_abs_interval_discrepancy_ms"],
        })
    progress.step_done()
display(pd.DataFrame(rows))
for problem in problems:
    print("NOT WRITTEN:", *problem)

## 2. Reward-aligned responses, by number of reward pulses

0 pulses is aligned to the no-reward cue onset, 1-4 pulses to reward onset (different events). The outcome goes to the frame whose start is nearest it (within half a frame), so time 0 is that frame's start. "Cells" are all Cellpose ROIs; Suite2p's `iscell` classifier is not used to exclude any. Each cell is averaged over its trials first, so each cell counts once; the shading is the SEM across cells. The default views are robust z-scores (each cell's whole trace: (x - median) / (1.4826 · MAD)) with each trial's mean over the part of the window before the outcome subtracted, so cells with the same trend but different raw F or ΔF/F0 scale do not average out; a cell with MAD 0 has no scale, is left out and is listed. Set `MEASURES = ("F", "dF/F0")` for the raw views (raw F reflects the cells' brightness). Runs left out are listed with the reason.

A long window (e.g. (−10, 30) s) can include the previous or next trial's outcome in the baseline or the response when trials are closer together than the window. The table's `outcomes dropped` (`n_events_dropped`) shows outcomes lost at run ends or sync gaps (no complete window of imaged frames), and `no z baseline` (`n_trials_no_baseline`) the trials left out because no pre-outcome frame remained for the z baseline (they are not in the pulse-count columns); z measures need at least 2 frames before the outcome. File names carry the window (e.g. `_win-10to30s`).

In [ ]:
responses = {}
for variant in VARIANTS:
    pooled, runs, skipped = collect_pulse_responses(
        DERIVATIVES, SESSION_FIRST, SESSION_LAST, variant, WINDOW_S, PHASES, runs=RUNS,
        measures=MEASURES,
    )
    responses[variant] = (pooled, runs)
    print(f"{variant}: {len(runs)} runs, {sum(r['n_cells'] for r in runs)} cells")
    for r in runs:
        for view, cells in r["flat_cells"].items():
            if cells:
                where = f"{r['session']} {r['run']} {view}"
                print(f"  {where}: ROIs {cells} have MAD 0; z is NaN")
        if r["n_trials_no_baseline"]:
            print(f"  {r['session']} {r['run']}: {r['n_trials_no_baseline']} trials without a z "
                  "baseline left out")
    for session, run, reason in skipped:
        print(f"  LEFT OUT {session} {run}: {reason}")
    if not runs:
        print(f"  {variant}: nothing to plot")
        continue
    counts = pd.DataFrame([{"session": r["session"], "run": r["run"], "cells": r["n_cells"],
                            **{f"{p} pulses": r["n_trials"].get(p, 0) for p in range(5)},
                            "outcomes dropped": r["n_events_dropped"],
                            "no z baseline": r["n_trials_no_baseline"]} for r in runs])
    display(counts)
    fig = plot_pulse_responses(
        pooled, runs[0]["offsets"], runs[0]["period_s"], x_units=X_UNITS,
        measures=MEASURES,
        title=f"{SUBJECT}, {SCOPE}, {variant}: mean ± SEM across cells",
    )
    name = pulse_responses_png_name(FILE_TAG, variant, MEASURE_TAG, X_UNITS, WINDOW_S)
    fig.savefig(FIGURES / name, dpi=200)
    plt.show()

## 2b. Background: the mean of all pixels that are not part of an ROI

A check for a **field-wide** effect. All pixels that are in no Cellpose ROI (cell or not) and inside the registration's valid crop are treated as one large ROI. Its F in each frame is the plain mean of those pixels (Suite2p's ROI mean with uniform weights), with no neuropil subtraction, for the green and the red channel. It is aligned to the outcomes and split by number of reward pulses exactly like the cells (same frame times, window, phases, 0 pulses at the no-reward cue, same robust z with each trial's pre-outcome mean subtracted), so it is directly comparable. If the background responds like the cells, the cells' responses may reflect something field-wide (blood flow, neuropil, motion, a diffuse neuromodulator) rather than the cells themselves; if it is flat, they do not.

There is one background trace per run, so the figure shows the mean ± SEM across runs (n in the legend; thin lines are single runs) for raw F (the main view) and robust-z F, with the cells' robust-z F of the same runs beneath on the same y limits. With a single run there is no SEM across runs, so the legend then says the band is the SEM across that run's trials instead. A run whose background is constant has no z scale; it is listed and is not in the z means.

Caveats: out-of-focus neuropil is part of the background; pixels outside the registration crop (`yrange`, `xrange`) are excluded because they hold no data, but Suite2p takes the crop from the non-bad frames only, so on frames with large motion some edge pixels inside the crop can be invalid (this affects the cells too); `ROI_MARGIN_PX` > 0 also drops a ring around every ROI to reduce ROI-edge and neuropil contamination. The sanity-check PDF shows, per run, the mean images (green, red) with the included pixels shaded and the ROIs outlined; the first run is shown here. Traces are cached in the repo's `.cache/funcimg_background/` (the binaries are read once, in chunks).

In [ ]:
BACKGROUND = True  # False skips this section
ROI_MARGIN_PX = 0  # pixels around every ROI to leave out of the background as well
BACKGROUND_MEASURES = ("F", "F z")  # raw F (main view) and robust z; the background has no dF/F0

if BACKGROUND:
    from src.funcimg_analysis.collect import collect_background_responses, pool_runs
    from src.funcimg_visualization.plot_funcimg_background import (
        background_responses_png_name,
        plot_background_mask,
        plot_background_responses,
    )

    for variant in VARIANTS:
        bg_pooled, bg_runs, bg_skipped = collect_background_responses(
            DERIVATIVES, SESSION_FIRST, SESSION_LAST, variant, WINDOW_S, PHASES, runs=RUNS,
            measures=BACKGROUND_MEASURES, roi_margin_px=ROI_MARGIN_PX,
        )
        for session, run, reason in bg_skipped:
            print(f"  LEFT OUT (background) {session} {run}: {reason}")
        if not bg_runs:
            print(f"{variant}: no run with a background to plot")
            continue
        for r in bg_runs:
            for view, flat in r["flat_cells"].items():
                if flat:
                    print(f"  {r['session']} {r['run']} {view}: constant background (MAD 0); "
                          "z is NaN and the run is not in the z means")
        display(pd.DataFrame([{
            "session": r["session"], "run": r["run"], "background pixels": r["n_pixels"],
            "fraction of FOV": round(r["fraction_of_fov"], 3),
            **{f"{p} pulses": r["n_trials"].get(p, 0) for p in range(5)},
            "no z baseline": r["n_trials_no_baseline"],
        } for r in bg_runs]))
        # the cells' z of the same runs: reuse section 2 when it already has it
        if "F z" in MEASURES and variant in responses:
            cell_runs = responses[variant][1]
        else:
            _, cell_runs, cell_skipped = collect_pulse_responses(
                DERIVATIVES, SESSION_FIRST, SESSION_LAST, variant, WINDOW_S, PHASES, runs=RUNS,
                measures=("F z",),
            )
            for session, run, reason in cell_skipped:
                print(f"  LEFT OUT (cells) {session} {run}: {reason}")
        with_background = {(r["session"], r["run"]) for r in bg_runs}
        with_cells = {(r["session"], r["run"]) for r in cell_runs}
        if with_background != with_cells:
            print(f"  runs differ: background only {sorted(with_background - with_cells)}; "
                  f"cells only {sorted(with_cells - with_background)}; "
                  "the cells' row uses the runs that have both")
        shared = [r for r in cell_runs if (r["session"], r["run"]) in with_background]
        cells_z = pool_runs(shared, ("F z",)) if shared else None
        fig = plot_background_responses(
            bg_pooled, cells_z, bg_runs[0]["offsets"], bg_runs[0]["period_s"], x_units=X_UNITS,
            per_run=[r["means"] for r in bg_runs], measures=BACKGROUND_MEASURES,
            title=f"{SUBJECT}, {SCOPE}, {variant}: background (mean ± SEM across runs) vs cells",
        )
        name = background_responses_png_name(FILE_TAG, variant, X_UNITS, WINDOW_S)
        fig.savefig(FIGURES / name, dpi=200)
        plt.show()
        pdf_path = FIGURES / f"background_pixels_{FILE_TAG}_{variant}.pdf"
        with PdfPages(pdf_path) as pdf:
            for i, r in enumerate(bg_runs):
                fig = plot_background_mask(
                    r["mean_images"], r["mask"], r["labels"], title=f"{r['session']} {r['run']} ({variant})",
                )
                if i == 0:
                    display(fig)  # the first run, here; every run is in the PDF
                pdf.savefig(fig)
                plt.close(fig)
        print("Saved", pdf_path)


## 2c. Shared vs cell-specific signal: do cells carry signal beyond the background?

Section 2b asks whether the background *responds* to the outcome. This asks the question underneath: do the cells share signal with each other **beyond** what the background (all non-ROI pixels as one ROI) already explains, and does the reward response survive once the background is removed? Green and red are analysed the same way, each against its own channel's background; a field signal in the red (structural) channel points to blood flow or motion rather than the indicator.

**Preparation.** Every trace (each cell's raw F and neuropil-corrected ΔF/F0, each cell's own neuropil `Fneu`, and the background's F) is detrended by subtracting a running 10th percentile over `DETREND_WINDOW_S` (so a slow drift or bleaching does not create correlations), then robust-z-scored (median / MAD), so scales do not matter. Within half a window of either end of a run the baseline comes from a one-sided, truncated window and is less reliable. All correlations use every frame that is finite in all traces involved, including frames with no pyControl time (the question is about the traces, not the events). "Cells" are all ROIs unless `CELLS_ONLY`, which keeps ROIs with `iscell[:, 0] > 0`. A run with fewer than two ROIs is listed and left out.

**How to read the result. The shifted null only tests partial r = 0, and the background is an imperfect control.** The background is a noisy, spatially different measurement of the field signal, so regressing it out leaves part of the field in the cells and partial r can be positive for cells that share *nothing* but the field. A partial r above its shifted null is therefore **not by itself evidence of shared neural signal**. Read it together with (1) the neuropil-controlled partial r (the background *and* each cell's own neuropil and the other cells' mean neuropil regressed out of both sides), (2) the background's split-half reliability (the background pixels split into two checkerboard halves, r after detrending, stepped up by Spearman-Brown) and (3) the residual reward response below.

**Panels and what they mean.**
- *Scatter* (one point per cell, colour = run): r(cell, average of the **other** cells) on y against r(cell, background) on x, with the identity line. The comparison uses the leave-one-out average so the same noisy cell sits on both sides (one cell against a cleaner reference each time); the whole-FOV average is not used because it is mostly background pixels, so it would be the background again. The average of a few cells carries their private noise, which lowers r(cell, average); the **filled** markers divide it by the square root of the average's reliability (split halves of the other cells, Spearman-Brown, at most 1), **hollow** markers are the raw values. Field-only cells then lie on the identity line for any number of cells, up to the background's own noise (which lowers x). The short coloured ticks on the axes are the 95th percentile of the circular-shift chance level of each correlation. The text gives, per run, the median difference (y - x) corrected and raw, and a paired Wilcoxon p across cells, as a description only: cells within a run are not independent. Cells outside the axes are counted in the text.
- *Partial r histogram*: solid is given the background, dotted given the background plus neuropil. The dashed and dash-dotted vertical lines are the 95th percentile of each one's null: the same statistic with the other cells' average circularly shifted by a random offset of at least `MIN_SHIFT_S` (`N_SHIFTS` shifts, seeded per run), pooled over cells and shifts. The table's `null 95th` columns reduce the null to the run's median per shift, i.e. the level the median partial r has to exceed.
- *R² histogram*: variance of each detrended cell explained by `cell ~ a + b · background`.
- *Residual responses*: each cell's detrended trace minus `b · background` (the cell keeps its own offset), then exactly the same pulse-count alignment, window, phases and pooling as the cell z responses. `b` is fitted only on frames **outside** a window around every outcome (all pulse counts; `FIT_EXCLUDE_OUTCOMES`, `FIT_EXCLUDE_WINDOW_S`) and applied to all frames, so a reward response the background happens to share (or leaks into it) cannot pull the slope towards itself; the table reports the fraction of frames used. Original (detrended cells) and residual are side by side on shared y limits, in units of the **detrended cell's** robust scale (not re-z-scored by the smaller scale of the residual, which would inflate it). The table lists the mean response 0-2 s after the outcome per pulse count for both.

**Chance levels** are limited to what is drawn or tabulated (`NULL_KINDS`); the others (`"partial_background"`: background shifted instead, `"r_bg_avg"`) are opt-in because each costs a full pass. Each run's result is cached in `.cache/funcimg_shared/`, keyed on the content of every trace, the frame times, outcomes, settings and the analysis code, so re-running the notebook takes seconds.

**Caveats.** Detrending matters: a different window changes the correlations (the file names carry it). Removing a real diffuse signal (a neuromodulator acting on all cells and the neuropil) removes signal that belongs to the cells, so a vanishing residual does not prove the response is an artefact. Raw F is not neuropil-corrected whereas ΔF/F0 is, so the two measures need not agree. Cells within a run are not independent, and cells of different runs of one field of view may be the same neurons. Runs shorter than the detrend window or twice `MIN_SHIFT_S` are listed and left out, as are runs whose outcome windows leave too few frames to fit the slope.

In [ ]:
SHARED_SIGNAL = True  # False skips this section
DETREND_WINDOW_S = 60.0  # running 10th-percentile baseline subtracted from every trace
N_SHIFTS = 100  # circular shifts per chance level
MIN_SHIFT_S = 30.0  # smallest circular shift (s), in both directions
CELLS_ONLY = False  # True: only ROIs the Suite2p classifier accepts (iscell[:, 0] > 0)
SHARED_MEASURES = ("F z", "dF/F0 z")  # analysed in z units: raw F and ΔF/F0 of each channel
FIT_EXCLUDE_OUTCOMES = True  # fit the background's slope only outside the outcome windows
FIT_EXCLUDE_WINDOW_S = (-1.0, 5.0)  # left out around each outcome; None: WINDOW_S
# chance levels to compute; add "partial_background" or "r_bg_avg" for those (each costs a pass)
NULL_KINDS = ("r_bg", "r_loo", "partial_average", "partial_neuropil_average")

if SHARED_SIGNAL:
    from src.funcimg_analysis.collect_shared import collect_shared_signal, pool_shared_signal
    from src.funcimg_analysis.run_statistics import summary_row
    from src.funcimg_analysis.shared_signal import mean_response
    from src.funcimg_visualization.plot_funcimg_shared import (
        plot_partial_r_histograms,
        plot_r2_histograms,
        plot_residual_responses,
        plot_scatter_cell_vs_background,
        residual_png_name,
        settings_tag,
        shared_png_name,
    )

    shared_tag = settings_tag(
        DETREND_WINDOW_S, N_SHIFTS, MIN_SHIFT_S, CELLS_ONLY, FIT_EXCLUDE_OUTCOMES,
        FIT_EXCLUDE_WINDOW_S,
    )
    shared_raw = tuple(m.removesuffix(" z") for m in SHARED_MEASURES)
    for variant in VARIANTS:
        shared_runs, shared_skipped = collect_shared_signal(
            DERIVATIVES, SESSION_FIRST, SESSION_LAST, variant, WINDOW_S, PHASES, runs=RUNS,
            measures=SHARED_MEASURES, roi_margin_px=ROI_MARGIN_PX, cells_only=CELLS_ONLY,
            detrend_window_s=DETREND_WINDOW_S, n_shifts=N_SHIFTS, min_shift_s=MIN_SHIFT_S,
            null_kinds=NULL_KINDS, fit_exclude_outcomes=FIT_EXCLUDE_OUTCOMES,
            fit_exclude_window_s=FIT_EXCLUDE_WINDOW_S,
        )
        for session, run, reason in shared_skipped:
            print(f"  LEFT OUT (shared signal) {session} {run}: {reason}")
        if not shared_runs:
            print(f"{variant}: no run to analyse for shared signal")
            continue
        display(pd.DataFrame([{
            "session": r["session"], "run": r["run"], "channel": channel, "measure": measure,
            "slope fitted on": f"{100 * r['fit_fraction']:.0f}% of frames",
            **{k: round(v, 3) if isinstance(v, float) else v for k, v in summary_row(s).items()},
        } for r in shared_runs for (channel, measure), s in r["stats"].items()]))
        title = f"{SUBJECT}, {SCOPE}, {variant}"
        for kind, plot in (
            ("scatter", plot_scatter_cell_vs_background),
            ("partial_r", plot_partial_r_histograms),
            ("r2", plot_r2_histograms),
        ):
            fig = plot(shared_runs, shared_raw, title=title)
            fig.savefig(FIGURES / shared_png_name(kind, FILE_TAG, variant, shared_tag), dpi=200)
            plt.show()
        original, residual = pool_shared_signal(shared_runs, SHARED_MEASURES)
        window = shared_runs[0]["original"]
        offsets, period = window["offsets"], window["period_s"]
        fig = plot_residual_responses(
            original, residual, offsets, period, x_units=X_UNITS, measures=SHARED_MEASURES,
            title=f"{title}: detrended cells vs background removed (mean ± SEM across cells)",
        )
        fig.savefig(
            FIGURES / residual_png_name(FILE_TAG, variant, X_UNITS, WINDOW_S, shared_tag), dpi=200
        )
        plt.show()
        first, second = (mean_response(p, offsets, period) for p in (original, residual))
        display(pd.DataFrame([
            {"channel": view[0], "measure": view[1], "pulses": pulses,
             "original 0-2 s": round(value, 3),
             "background removed 0-2 s": round(second[view][pulses], 3)}
            for view, per_pulse in first.items() for pulses, value in sorted(per_pulse.items())
        ]))


## 3. The most responsive cells of each run

Two rankings, each its own PDF with one page per run: `top{N_TOP}_cells_by_p_cell_….pdf` ranks cells by Suite2p's classifier probability p(cell) (`iscell.npy` column 1; advisory, since Suite2p's built-in classifier is run on Cellpose ROIs, not tuned for them), and `top{N_TOP}_cells_by_max_dff_….pdf` by the largest green ΔF/F0 anywhere in the recording (a single bright frame can rank a cell high, so check the traces). Ties go to the lower ROI index; each trace's label shows its p(cell). Each trace is drawn clipped to its own `CLIP_PERCENTILES` range (display only; the ranking uses the unclipped values), and the rows are spaced so the clipped traces never overlap. Clipping flattens every peak above the 99th percentile (with the default `(1, 99)`), so the **tick label carries the ROI's true maximum ΔF/F0 over the whole run** (`ROI 12 (p=0.93, max 2.4)`), and a small tick on the x axis, in the ROI's colour, marks where that maximum falls when it is inside the frames shown. An ROI with no data in the frames shown (possible with `TRACES_MIDDLE_FRAMES`) is an empty row labelled "(no data in shown frames)". File names carry the display settings (e.g. `_frames_mid8000_clip1-99`), so a figure made with other settings is not overwritten. The first run of each is shown here.

In [ ]:
for variant in VARIANTS:
    found = variant_runs(DERIVATIVES, SESSION_FIRST, SESSION_LAST, variant, RUNS)
    if not found:
        print(f"{variant}: no registered runs found")
        continue
    for rank_by in RANKINGS:
        progress = ProgressLog(f"Top cells by {rank_by} ({variant})", len(found), print, 30)
        pdf_path = FIGURES / top_cells_pdf_name(
            N_TOP, rank_by, FILE_TAG, variant, TRACES_X_UNITS,
            middle_frames_shown=TRACES_MIDDLE_FRAMES, clip_percentiles=CLIP_PERCENTILES,
        )
        pdf = None  # opened on the first page, so a ranking with no plottable run writes no PDF
        try:
            for session, run_dir, plane_dir in found:
                try:
                    top = top_cells(plane_dir, run_dir, N_TOP, phases=PHASES, rank_by=rank_by)
                except (RunMismatchError, OutcomeParseError, FileNotFoundError) as error:
                    print(f"  LEFT OUT {session} {run_dir.name}: {error}")
                    progress.step_done(succeeded=False)
                    continue
                fig = plot_top_cells(
                    top, x_units=TRACES_X_UNITS, title=f"{session} {run_dir.name}", variant=variant,
                    clip_percentiles=CLIP_PERCENTILES, middle_frames_shown=TRACES_MIDDLE_FRAMES,
                )
                if pdf is None:
                    pdf = PdfPages(pdf_path)
                    display(fig)  # the first run, here; every run is in the PDF
                pdf.savefig(fig)
                plt.close(fig)
                progress.step_done()
        finally:
            if pdf is not None:
                pdf.close()  # also after an error, so the PDF is never left unfinished
        if pdf is None:
            print(f"{variant}: no run could be plotted by {rank_by}")
            continue
        print("Saved", pdf_path)
